# mtrl on Colab: smoke test

Checks that the post-training code runs on Colab: data build, LoRA SFT and RL for a few steps
each with Qwen3.5-2B, resume from checkpoint.

Before running: **Runtime → Change runtime type** → pick a GPU. **L4** is the recommended card
(bf16, like the local server); **A100 40/80 GB** for bigger runs; **T4** works only in fp16,
which may overflow with Qwen3.5. Then run the cells in order (about 20–30 minutes, mostly
installs and downloads).

The env config is picked from the GPU in step 3b (`scripts/pick_env.py colab`):

| GPU | env config | dtype |
|---|---|---|
| T4 | `colab_T4_16G` | fp16 |
| L4 | `colab_L4_24G` | bf16 |
| A100 40 GB | `colab_A100_40G` | bf16 |
| A100 80 GB | `colab_A100_80G` | bf16 |

All four share `configs/env/_colab.yaml` (Drive paths, venvs). Another GPU stops at step 3b
with an error instead of guessing.

| Step | Checks |
|---|---|
| 1–2 | GPU; Drive mounted (runs, checkpoints, HF cache are stored there) |
| 3 | training packages install into the runtime (transformers 5.x, peft); env config picked |
| 4 | the MetricX scorer installs into its own venv (it needs transformers 4.x) |
| 5 | training data builds (mtdata download, cleaning) for en-is and en-kk |
| 6 | unit tests pass |
| 7–8 | SFT and RL run, losses are finite, validation and checkpoints work |
| 9 | resume from checkpoint works |

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Drive and code

Runs, checkpoints and the Hugging Face cache go to `MyDrive/mtrl/`, so a disconnect loses nothing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/mtrl/hf /content/drive/MyDrive/mtrl/runs /content/drive/MyDrive/mtrl/ckpt

BRANCH = 'train/merge-ym'   # change to 'main' once the PR is merged
!rm -rf /content/mtrl && git clone -q -b {BRANCH} https://github.com/ETHz-DL-team29/mtrl.git /content/mtrl
%cd /content/mtrl
!git log --oneline -1

In [ ]:
import os
os.environ['PYTHONPATH'] = '/content/mtrl/src'   # the scorer venv imports mtrl from here too
# Optional: a Hugging Face token avoids download rate limits (Colab: key icon -> secret HF_TOKEN).
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    print('no HF_TOKEN secret; continuing without')

## 3. Training packages (Colab runtime)

Qwen3.5 needs transformers 5.x. Colab's preinstalled torch is kept.

In [ ]:
!pip install -q "transformers>=5" "peft>=0.17" fasttext-numpy2-wheel sacrebleu pyyaml nvidia-ml-py huggingface_hub pytest
import torch, transformers, peft
print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__)
assert transformers.__version__.startswith('5'), 'need transformers 5.x for Qwen3.5'
assert torch.cuda.is_available()

## 3b. Pick the env config for this GPU


In [ ]:
import subprocess
env_name = subprocess.run(['python', 'scripts/pick_env.py', 'colab'], capture_output=True, text=True)
print(env_name.stdout, env_name.stderr)
assert env_name.returncode == 0, 'no env config for this GPU (see the message above)'
os.environ['MTRL_ENV'] = env_name.stdout.strip()
open('.env', 'w').write(f"MTRL_ENV={os.environ['MTRL_ENV']}\n")
!cat configs/env/{os.environ['MTRL_ENV']}.yaml

## 4. Scorer venv (MetricX) and mtdata venv

MetricX / COMET need transformers 4.x and numpy<2, which conflict with step 3, so the scorer
gets its own venv (`python.score` in the env config). The trainer starts it automatically.
This installs a second torch and takes several minutes.

In [ ]:
!python -m venv /content/score-venv
!/content/score-venv/bin/pip install -q -r setup/requirements-score.txt
!/content/score-venv/bin/python -c "import torch, transformers; print('score venv: torch', torch.__version__, 'transformers', transformers.__version__, 'cuda', torch.cuda.is_available())"

!python -m venv /content/mtdata-venv
!/content/mtdata-venv/bin/pip install -q "mtdata[hf]==0.5.1"
!/content/mtdata-venv/bin/mtdata --version

## 5. Data

fastText language ID, then the training / dev / test data for the two directions the smoke
tests use (en-is, en-kk). Same code and config as on the server (`configs/train_data.yaml`);
the full set is `python scripts/prepare_train_data.py` without `--directions`.

In [ ]:
!mkdir -p data/models && [ -f data/models/lid.176.bin ] || wget -q -O data/models/lid.176.bin https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin
!python scripts/prepare_train_data.py --directions en-is en-kk
!wc -l data/processed/en-is/*.jsonl data/processed/en-kk/*.jsonl

## 6. Unit tests

In [ ]:
!python -m pytest -q tests 2>&1 | tail -3

## 7. SFT smoke test (6 steps)

Watch for: `loss` finite (on T4 / fp16 an overflow shows as `nan` or `skipped_nonfinite > 0`), `vram_gb`,
and a `[val]` line after steps 3 and 6.

In [ ]:
!python scripts/train.py configs/train/quick_sft.yaml 2>&1 | grep -E "^\[(sft|val|train)\]|Error|error" | cut -c1-300

## 8. RL smoke test (6 steps)

Starts the MetricX service from the venv on the first step (model download on first use).
Watch for: `reward` and `metricx` values, `grad_norm` finite, `[val]` lines.
If it fails, the scorer log is in `runs/quick_rl/scorer.log` on Drive.

In [ ]:
!python scripts/train.py configs/train/quick_rl.yaml 2>&1 | grep -E "^\[(rl|val|train|scorer)\]|Error|error" | cut -c1-300

## 9. Resume

Raising the step count and re-running the same config must continue from the last checkpoint (step 6), not from 0.

In [ ]:
!sed -i 's/steps: 6/steps: 8/' configs/train/quick_sft.yaml
!python scripts/train.py configs/train/quick_sft.yaml 2>&1 | grep -E "resumed|^\[sft\]" | cut -c1-200
!git checkout configs/train/quick_sft.yaml

## 10. Summary to report back

In [ ]:
import json, pathlib
runs = pathlib.Path('/content/drive/MyDrive/mtrl/runs')
for r in ['quick_sft', 'quick_rl']:
    log = [json.loads(l) for l in (runs / r / 'train_log.jsonl').read_text().splitlines()]
    val = [json.loads(l) for l in (runs / r / 'val_log.jsonl').read_text().splitlines()] if (runs / r / 'val_log.jsonl').exists() else []
    print(r, '| steps', len(log), '| max vram_gb', max(x.get('vram_gb', 0) for x in log),
          '| last sec/step', log[-1].get('sec'), '| nonfinite', sum(x.get('skipped_nonfinite', 0) for x in log),
          '| last val', {k: v for k, v in (val[-1] if val else {}).items() if k in ('val_chrf', 'val_gate_pass', 'val_reward', 'error')})